# Section 1 Assignment: Car prices 🚗

In Section 1 we prepared and explored the **churn** data. In this assignment you do the **same steps on a new problem**, so you practice *why* each step is needed, not just *how* to type it.

**The story:** you work at *AutoSell*, a website where people sell used cars. Sellers don't know what price to ask for. Later we will build a model that suggests a price. Today we get the data ready and find out what makes a car expensive.

**The data:** about 11,000 cars with their make, engine, size, style and price (`msrp`, the price in US dollars).

**Your data is unique.** The first cell gives you your own random sample of the cars, chosen by your college ID. Your numbers will be different from your friends' numbers, so copying answers won't work. Helping each other understand the steps is fine. 🙂

## Before you start (read this!)

1. **File → Save a copy in Drive.** If you skip this, your work is not saved.
2. **Write your college ID** in the first code cell below, then run it.
3. **Go step by step.** Every step explains what to do. Write your code where you see `...`, then run the `grader.check(...)` cell under it.
4. **About `grader.check`:** ✅ means your answer has the **right form** (for example, a number where we expect a number). It does **not** mean the answer is correct. Your real grade is calculated after the deadline.
5. **Rules:**
    - Don't rename the variables we ask for (`n_rows`, `hp_median`, ...). The grader looks for these exact names.
    - Don't change or delete the setup cell or the `grader.check` cells.
    - You don't need plots or any extra libraries. Use only what the steps ask for.
6. **How to submit** is explained at the end of the notebook.

In [ ]:
# write your college ID between the quotes, for example: "202202"
STUDENT_ID = ...

# ---- setup: do not change anything below this line ----
if not isinstance(STUDENT_ID, str) or not STUDENT_ID.isdigit():
    raise ValueError('Write your college ID on the first line, between quotes, for example: "202202"')
import os
if not os.path.exists("car_prices.csv"):   # first run on Colab: get the data and the checks
    !git clone -q --depth 1 https://github.com/omaramir2001/ml-assignments.git
    !cp -r ml-assignments/section1/. .
    %pip install -q otter-grader
import otter
import numpy as np
import pandas as pd
grader = otter.Notebook()
df = pd.read_csv("car_prices.csv").sample(n=8000 + int(STUDENT_ID) % 3000, random_state=int(STUDENT_ID)).reset_index(drop=True)
print(f"Your data is ready: {len(df)} cars.")

In [ ]:
df.head()

# Part 1: Data preparation

## Step 1: How big is your data?

Like in the section, start by checking the size of the table. `df.shape` gives `(rows, columns)`.

Store the number of rows (cars) in `n_rows` and the number of columns in `n_cols`.

In [ ]:
n_rows = ...
n_cols = ...
print(n_rows, n_cols)

In [ ]:
grader.check("q1")

## Step 2: Make column names and values look uniform

Look at `df.head()` again. Some column names have capital letters and spaces (`Engine HP`, `Transmission Type`), and some values too (`MANUAL`, `rear wheel drive`). Fix both, **exactly like in the section**:

1. Make the column names lowercase and replace spaces with underscores.
2. Find the text columns with `select_dtypes(exclude='number')`, then do the same to their values.

After this step, `Transmission Type` becomes `transmission_type` and `rear wheel drive` becomes `rear_wheel_drive`.

⚠️ Use `select_dtypes(exclude='number')` like the section does, not `df.dtypes == 'object'`. The second one silently finds no columns on newer Pandas versions.

In [ ]:
# 1. column names
df.columns = ...

# 2. values of the text columns
text_columns = ...
for c in text_columns:
    df[c] = ...

df.head()

In [ ]:
grader.check("q2")

Now check the type of every column, like in the section:

In [ ]:
df.dtypes

The numbers already look like numbers this time (`int64` or `float64`). So unlike `totalcharges` in the section, no column needs converting. The problem in this data is **missing values**.

## Step 3: Find the missing values

Count the missing values in each column. `isnull()` marks every missing cell, and `sum()` counts them per column.

Then answer two questions:
- Which column has the **most** missing values? Store its name as text in `most_missing_column`, for example `most_missing_column = "make"`.
- How many values are missing in `engine_cylinders`? Store the number in `n_missing_cylinders`.

Hint: you can read the answers from the output, or let Pandas find them: `.idxmax()` gives the name of the biggest value in a list of counts.

In [ ]:
print(df.isnull().sum())

most_missing_column = ...
n_missing_cylinders = ...

In [ ]:
grader.check("q3")

## Step 4: Missing cylinders: look before you fill

In the section, before filling the missing `totalcharges` we **looked at those customers first** and noticed their `tenure` was 0. Do the same here. Run the cell below to see the cars with missing cylinders.

```python
df[df.engine_cylinders.isnull()][['make', 'model', 'engine_fuel_type']]
```

**Question:** which **make** appears most often in that list? Store it as text in `missing_cylinders_make`. You can count by eye, or use `.value_counts()` on the `make` column of those rows and take `.idxmax()`.

Then fill the missing cylinders with **0** (see the explanation after this step for why 0 is right here).

In [ ]:
print(df[df.engine_cylinders.isnull()][['make', 'model', 'engine_fuel_type']])

missing_cylinders_make = ...
df.engine_cylinders = ...

In [ ]:
grader.check("q4")

**Why 0?** Most of those cars are **Mazda RX-7 and RX-8**. They use a *rotary engine*, which has no cylinders at all. The rest are **electric** cars, which have no engine cylinders either. So these cars really have 0 cylinders. It's the same idea as the 11 new customers in the section, whose total charge really was 0.

## Step 5: Missing horsepower: when 0 is the wrong choice

`engine_hp` also has missing values. Can we fill them with 0 too? **No.** Every car has some horsepower, and a car with 0 HP doesn't exist. Filling with 0 would teach a model something false.

A safer choice is the **median**: the middle value when you sort all the horsepowers. It isn't pulled up by a few super-powerful sports cars the way the mean is.

1. Store the median of `engine_hp` in `hp_median`.
2. Fill the missing values of `engine_hp` with `hp_median`.

In [ ]:
hp_median = ...
df.engine_hp = ...
print(hp_median)

In [ ]:
grader.check("q5")

## Step 6: Fill the rest and check

Three columns still have missing values:

- `market_category`: a missing value means the car has **no special category** (not luxury, not performance, ...). Fill it with the text `'unknown'`.
- `engine_fuel_type` and `number_of_doors`: only a handful are missing. Fill them with the **most frequent value** of their column. `.mode()` gives the most frequent values and `[0]` takes the first one, for example `df.number_of_doors.mode()[0]`.

Finally, do the **final check** from the section: count all missing values in the whole table and store the number in `total_missing`. It should be 0.

In [ ]:
df.market_category = ...
df.engine_fuel_type = ...
df.number_of_doors = ...

total_missing = ...
print(total_missing)

In [ ]:
grader.check("q6")

# Part 2: The target

In the section the target was `churn` (yes/no), and we turned it into 1/0. Here the target is the **price**, `msrp`, which is already a number. It needs a different kind of preparation.

## Step 7: The price has a long tail

1. Store the mean price in `price_mean` and the median price in `price_median`.
2. Compare them. The mean is much bigger than the median. Why? A few very expensive cars (some cost over \$1,000,000!) pull the mean up, while most cars are much cheaper. This shape is called a **long tail**, and it confuses models.
3. The usual fix is the **logarithm**, which squeezes big numbers together: 1,000,000 becomes about 13.8 and 30,000 becomes about 10.3. Create a new column `log_msrp` with `np.log1p(df.msrp)`. (`log1p` means "log of 1 + the value", which avoids `log(0)`.)

In [ ]:
price_mean = ...
price_median = ...
print(price_mean, price_median)

df['log_msrp'] = ...

In [ ]:
grader.check("q7")

## Step 8: A yes/no version of the price: `is_expensive`

Some tools, like **mutual information** from the section, only work with **categories**. Price is a number, and almost every car has a different price, so we can't use it directly. The trick: ask a yes/no question instead.

> **Is this car more expensive than the median price?**

1 means yes (expensive) and 0 means no, just like churn in the section. Create the column `is_expensive` the same way we created churn's 1/0 column: a comparison gives `True`/`False`, and `.astype(int)` turns them into 1/0. Use `price_median` from the last step.

Then store the share of expensive cars (the mean of the column) in `expensive_share`.

In [ ]:
df['is_expensive'] = ...
expensive_share = ...
print(expensive_share)

In [ ]:
grader.check("q8")

You should get a little **under 0.5**. That makes sense: the median splits the cars in half, and the few cars that cost exactly the median price count as "not expensive".

# Part 3: The validation framework

## Step 9: Put part of the data aside

Exactly like the section: split the data into **80% train** and **20% validation**, with `random_state=42`. Call the parts `df_train` and `df_val`.

Then store their sizes in `n_train` and `n_val`.

⚠️ The split makes copies of `df`. Run it **after** all the cleaning steps above, so the copies are clean.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = ...
n_train = ...
n_val = ...
print(n_train, n_val)

In [ ]:
grader.check("q9")

**From now on, we only explore `df_train`.**

# Part 4: EDA: Exploratory Data Analysis

## Step 10: Average price, and the price inside a group

1. Store the global average price in `df_train` in `global_mean_price`.
2. Like the churn rate per contract in the section, use `groupby` to get the average price **and** the count for every make. Store the table in `make_prices`.

In [ ]:
global_mean_price = ...
make_prices = ...

make_prices.sort_values('mean', ascending=False).head(10)

In [ ]:
grader.check("q10")

## Step 11: Why `count` matters

Look at the top of your table. Which make is first, and **how many cars** does it have? A make with only a few cars can top the list by chance. That's why the section said a number from a big group is more reliable.

Keep only the makes with **at least 100 cars**, and store the name of the most expensive one among them in `top_make` (as text).

Hint: `make_prices[make_prices['count'] >= 100]` keeps those rows. Then `['mean'].idxmax()` gives the make with the highest average.

In [ ]:
big_makes = ...
top_make = ...
print(top_make)

In [ ]:
grader.check("q11")

## Step 12: The "expensive rate" inside a group

In the section, we compared the **churn rate** inside each group with the global churn rate. Do the same with `is_expensive`:

1. Store the global share of expensive cars in `df_train` in `global_expensive_rate`.
2. Use `groupby` on `vehicle_size` to get the share of expensive cars in each size. Store the result in `size_expensive_rate`. Use `.mean()` only, not `agg`, so you get one number per size.
3. Store the size with the **highest** expensive rate in `most_expensive_size` (as text).

**Think about it:** which sizes are above the global rate, and which are below?

In [ ]:
global_expensive_rate = ...
size_expensive_rate = ...
most_expensive_size = ...

print(global_expensive_rate)
size_expensive_rate

In [ ]:
grader.check("q12")

# Part 5: Feature importance

## Step 13: Mutual information with `is_expensive`

Now we use `is_expensive` for what we created it for. In the section we wrote a small function that compares one column with `churn`, then used `apply` to run it on every categorical column. Do the same, but compare with **`df_train.is_expensive`**.

The list of categorical columns is given. Notice what's **not** in it:
- `model` is left out. It has hundreds of different values, almost one per car, like `customerid` in the section.
- `number_of_doors` is a number, but it only has 3 values (2, 3, 4), so we treat it as a category, like `seniorcitizen` in the section.

Store the scores in `mi`, and the name of the column with the highest score in `top_mi_column` (as text).

In [ ]:
from sklearn.metrics import mutual_info_score

categorical = ['make', 'engine_fuel_type', 'transmission_type', 'driven_wheels',
               'number_of_doors', 'market_category', 'vehicle_size', 'vehicle_style']

def mutual_info_expensive_score(series):
    ...

mi = ...
top_mi_column = ...

mi.sort_values(ascending=False)

In [ ]:
grader.check("q13")

## Step 14: Correlation with the price

For the numerical columns we use **correlation**, like the section. The target here is a number, so we don't need the yes/no trick. We can correlate directly with **`log_msrp`**.

1. Use `corrwith` to get the correlation of each numerical column with `df_train.log_msrp`. Store it in `corr`.
2. Store the name of the column whose correlation is **furthest from 0** in `top_corr_column` (as text). "Furthest from 0" counts in either direction, so use `corr.abs()` before `.idxmax()`.

In [ ]:
numerical = ['year', 'engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']

corr = ...
top_corr_column = ...

corr

In [ ]:
grader.check("q14")

**Think about it (not graded):** you probably found that `year` matters even more than horsepower. Why? The data includes many cars from the 1990s, and most of them are listed at only about \$2,000. New cars cost much more. Questions like this are exactly what EDA is for: they make you look closer at your data.

## Summary
- The price has a long tail, so we use `log_msrp` for numbers and `is_expensive` for yes/no questions.
- How we fill missing values depends on what the missing value **means**: 0 for cylinders (rotary and electric engines really have none), the median for horsepower (no car has 0 HP), and `'unknown'` for the market category.
- With small groups, look at the **count** before trusting an average.

# How to submit ✅

1. **Runtime → Restart session and run all.** Wait until every cell has run, and make sure there are **no red errors**. After the deadline, the grader runs your notebook from top to bottom in exactly this way, so an error stops every answer after it.
2. **File → Download → Download .ipynb.**
3. **Rename the file to your college ID**, for example `202202.ipynb`. Use the same ID you wrote in the first cell. If they don't match, your submission is flagged.
4. **Upload it in the Google Form** your instructor shared, before the deadline.

Good to know:
- Upload the `.ipynb` file itself, not a link and not a `.py` file.
- Your grade is calculated after the deadline, from the variables you created in each step.